# Week 1 — Retrieval Models: ALS & Item2Vec
Train ALS and Item2Vec, evaluate on small_matrix, benchmark FAISS.


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

## 1. Load & Split Data

In [2]:
from src.data.preprocessing import (
    load_big_matrix, load_small_matrix,
    split_big_matrix, build_ground_truth
)

big   = load_big_matrix()
small = load_small_matrix()

train, val, test = split_big_matrix(big)
ground_truth     = build_ground_truth(small)

# Evaluation users = intersection of small_matrix users and train users
train_users = set(train['user_id'].unique())
eval_users  = [u for u in ground_truth if u in train_users]
print(f'Eval users (in train & small_matrix): {len(eval_users):,}')

Split sizes  train: 8,771,564  val: 1,879,621  test: 1,879,621
Ground truth built for 1,411 users from small_matrix
Eval users (in train & small_matrix): 1,411


## 2. ALS

In [3]:
from src.retrieval.als import ALSRetriever
from src.evaluation.metrics import evaluate_retrieval, print_metrics

als = ALSRetriever(factors=128, iterations=30, alpha=40)
als.fit(train)
als.save('../models/als.pkl')

c:\anaconda3\envs\stock\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/30 [00:00<?, ?it/s]

ALS trained: 128 factors, 30 iterations
ALS model saved to ../models/als.pkl


In [4]:
# Evaluate ALS on small_matrix
als_recs = als.recommend_batch(eval_users, n=50)
als_scores = evaluate_retrieval(als_recs, ground_truth, k_list=[10, 20, 50])
print_metrics(als_scores, 'ALS')


  ALS
  NDCG@10        0.0000
  NDCG@20        0.0000
  NDCG@50        0.0000
  Recall@10      0.0000
  Recall@20      0.0000
  Recall@50      0.0000



## 3. Item2Vec

In [5]:
from src.retrieval.item2vec import Item2VecRetriever

i2v = Item2VecRetriever(vector_size=128, window=1000, epochs=10)
i2v.fit(train)
i2v.save('../models/item2vec.pkl')

Sequences built for 7,174 users (threshold=0.5, total interactions=5,685,497)
Training Word2Vec on 7,174 sequences (vector_size=128, window=1000)...


KeyboardInterrupt: 

In [ ]:
# Evaluate Item2Vec on small_matrix
i2v_eval_users = [u for u in eval_users if u in i2v.user_embeddings]
i2v_recs   = i2v.recommend_batch(i2v_eval_users, n=50)
i2v_scores = evaluate_retrieval(i2v_recs, ground_truth, k_list=[10, 20, 50])
print_metrics(i2v_scores, 'Item2Vec')

## 4. FAISS Index + Latency Benchmark

In [ ]:
from src.indexing.faiss_index import FAISSIndex

# Build FAISS index from ALS item embeddings
item_ids   = list(als.idx2item.values())
item_embs  = als.get_item_embeddings()
user_embs  = als.get_user_embeddings()

# ---- Exact index ----
idx_exact = FAISSIndex(mode='exact')
idx_exact.build(item_ids, item_embs)
bench_exact = idx_exact.benchmark(user_embs, k=100, n_queries=200)

# ---- Approximate index ----
idx_approx = FAISSIndex(mode='approx', n_list=100)
idx_approx.build(item_ids, item_embs)
bench_approx = idx_approx.benchmark(user_embs, k=100, n_queries=200)

idx_exact.save('../models/faiss_als_exact.index', '../models/faiss_als_exact_meta.json')
idx_approx.save('../models/faiss_als_approx.index', '../models/faiss_als_approx_meta.json')

In [ ]:
# FAISS-backed recommendations (exact)
def faiss_recommend_batch(faiss_idx, user_ids, user2idx, user_embs_arr, n=100):
    recs = {}
    for uid in user_ids:
        if uid not in user2idx:
            continue
        vec = user_embs_arr[user2idx[uid]]
        recs[uid] = faiss_idx.search(vec, k=n)
    return recs

faiss_recs   = faiss_recommend_batch(idx_exact, eval_users, als.user2idx, user_embs, n=50)
faiss_scores = evaluate_retrieval(faiss_recs, ground_truth, k_list=[10, 20, 50])
print_metrics(faiss_scores, 'ALS + FAISS (exact)')

## 5. Comparison Table

In [ ]:
import pandas as pd

rows = [
    {'Model': 'ALS',              **als_scores},
    {'Model': 'Item2Vec',         **i2v_scores},
    {'Model': 'ALS+FAISS(exact)', **faiss_scores},
]
results_df = pd.DataFrame(rows).set_index('Model')
results_df = results_df[sorted(results_df.columns)]
results_df = results_df.round(4)
print(results_df.to_string())
results_df.to_csv('../experiments/week1_results.csv')
print('\nSaved to experiments/week1_results.csv')

In [ ]:
# Visualise Recall@K and NDCG@K
ks = [10, 20, 50]
models = {'ALS': als_scores, 'Item2Vec': i2v_scores}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name, scores in models.items():
    axes[0].plot(ks, [scores[f'Recall@{k}'] for k in ks], marker='o', label=model_name)
    axes[1].plot(ks, [scores[f'NDCG@{k}']   for k in ks], marker='o', label=model_name)

axes[0].set_title('Recall@K'); axes[0].set_xlabel('K'); axes[0].legend()
axes[1].set_title('NDCG@K');   axes[1].set_xlabel('K'); axes[1].legend()
plt.tight_layout()
plt.savefig('../experiments/week1_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
print('\n=== FAISS Latency Benchmark ===')
print(f"Exact  — mean: {bench_exact['latency_ms_mean']:.2f}ms   p95: {bench_exact['latency_ms_p95']:.2f}ms")
print(f"Approx — mean: {bench_approx['latency_ms_mean']:.2f}ms  p95: {bench_approx['latency_ms_p95']:.2f}ms")
print('Target: < 10ms per query')